# Filesystems

> For filesystems that store video, text, and more

In [ ]:
# | default_exp filesystems


In [ ]:
# | export
from typing import Any
from pydantic import BaseModel
from dataclasses import dataclass, field
from enum import Enum

In [ ]:
# | hide
from nbdev.showdoc import *

In [ ]:
# | export
def foo():
    pass

In [ ]:
# | export
class File(BaseModel):
    uri: str


class User(BaseModel):
    id: str


class FileSystemType(Enum):
    LOCAL = "local"
    R2 = "r2"


class FilePathType(Enum):
    USER_ID = "user_id"


@dataclass
class AbstractFileSystem:
    base_path: str = field(default="")

    def build_file_path(self, user: User, file_path_type: FilePathType) -> str:
        """Build the file path according to the inputted objects and rules.
        Does not create resources, just a string formatter"""
        raise NotImplementedError("This method should be implemented by subclasses")

    def create_folder(self, path: str) -> str:
        """Creates folders at the specified location."""
        raise NotImplementedError("This method should be implemented by subclasses")

    def validate_access(self, user: User, path: str) -> bool:
        """Check if the user has access to the file at the specified path."""
        raise NotImplementedError("This method should be implemented by subclasses")

    def check_exists(self, path: str) -> bool:
        """Check if the file exists at the specified path."""
        raise NotImplementedError("This method should be implemented by subclasses")

    def create_file(self, path: str, content: bytes) -> File:
        """Creates files at specified location.
        Throws and error if the location doesn't exist, or the user doesn't have access."""
        raise NotImplementedError("This method should be implemented by subclasses")

    def read_file(self, path: str) -> File:
        raise NotImplementedError("This method should be implemented by subclasses")

    def update_file(self, path: str, content: bytes) -> File:
        raise NotImplementedError("This method should be implemented by subclasses")

    def delete_file(self, path: str) -> File:
        raise NotImplementedError("This method should be implemented by subclasses")

In [ ]:
# | export
import os
import tempfile


@dataclass
class LocalFileSystem(AbstractFileSystem):
    base_path: str = field(
        default=tempfile.mkdtemp(),
        metadata={"help": "The base path for the local file system."},
    )

    def build_file_path(
        self: AbstractFileSystem, user: User, file_path_type: FilePathType
    ) -> str:
        if file_path_type == FilePathType.USER_ID:
            return f"/Users/max/Documents/product_horse/temp/{user.id}"

    def create_folder(self: AbstractFileSystem, path: str) -> str:
        os.makedirs(path, exist_ok=True)
        return path

    def validate_access(self: AbstractFileSystem, user: User, path: str) -> bool:
        # Simplified access validation: checks if path starts with user's directory
        return path.startswith(f"/home/{user.id}")

    def check_exists(self: AbstractFileSystem, path: str) -> bool:
        return os.path.exists(path)

    def create_file(self: AbstractFileSystem, path: str, content: bytes) -> File:
        if not self.check_exists(os.path.dirname(path)) or not self.validate_access(
            User(id="default"), path
        ):
            raise FileNotFoundError("Location doesn't exist or access denied")
        with open(path, "wb") as f:
            f.write(content)
        return File(uri=path)

    def read_file(self: AbstractFileSystem, path: str) -> File:
        if not os.path.isfile(path):
            raise FileNotFoundError("File does not exist")
        return File(uri=path)

    def update_file(self: AbstractFileSystem, path: str, content: bytes) -> File:
        if not os.path.isfile(path):
            raise FileNotFoundError("File does not exist")
        return File(uri=path)

    def delete_file(self: AbstractFileSystem, path: str) -> File:
        if not os.path.isfile(path):
            raise FileNotFoundError("File does not exist")
        os.remove(path)
        return File(uri=path)

In [ ]:
from hypothesis import strategies as st
from hypothesis.stateful import (
    RuleBasedStateMachine,
    rule,
    precondition,
    Bundle,
    consumes,
)
from product_horse.core import run_test_case_with_pdb
# import tempfile
# import shutil
# from pathlib import Path


class FileSystemStateMachine(RuleBasedStateMachine):
    def __init__(self):
        super().__init__()
        self.file_system = LocalFileSystem()
        self.user = User(id="test_user")
        self.file_count = 0

    # def teardown(self):
    #     shutil.rmtree(self.temp_dir)

    files = Bundle("files")

    @rule(target=files, filename=st.text(min_size=1), content=st.binary(min_size=1))
    def create_file(self, filename: str, content: bytes) -> File:
        path = self.file_system.create_file(filename, content)
        assert self.file_system.check_exists(str(path))
        self.file_count += 1
        return path

    @rule(file=consumes(files))
    @precondition(lambda self: self.file_count > 0)
    def delete_file(self, file: File):
        self.file_system.delete_file(file.uri)

    @rule(file=files)
    @precondition(lambda self: self.file_count > 0)
    def read_file(self, file: File):
        assert self.file_system.read_file(file.uri).uri == str(file)

    @rule(file=files, content=st.binary(min_size=1))
    @precondition(lambda self: self.file_count > 0)
    def update_file(self, file: File, content: bytes):
        self.file_system.update_file(file.uri, content)
        assert self.file_system.check_exists(file.uri)


TestFileSystem = FileSystemStateMachine.TestCase
run_test_case_with_pdb(TestFileSystem, pdb_flag=False)

> /var/folders/qj/d65y3xj97dj5gmv5gpb9bq3m0000gn/T/ipykernel_92533/616533719.py(26)create_file()
     24     def create_file(self: AbstractFileSystem, path: str, content: bytes) -> File:
     25         if not self.check_exists(os.path.dirname(path)) or not self.validate_access(User(id="default"), path):
---> 26             raise FileNotFoundError("Location doesn't exist or access denied")
     27         with open(path, 'wb') as f:
     28             f.write(content)

False


Ran 1 test in 20.656s

FAILED (errors=1)


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore